# 25. Agent tool outputs: a matched set from AgentDojo

The introduction names emails, web pages and tool outputs; the two document sources so far append an instruction to short content. This notebook evaluates every scorer on AgentDojo's own tool outputs: emails, calendar entries, drive files, Slack messages and web pages, bills and transactions, hotel, restaurant and car-rental listings, rendered exactly as the benchmark's agents read them. Each injection slot appears once with its benign default and once per (attack style, injection task) pair with AgentDojo's own attack string; every other record in the benign environments is a further benign document. Six attack styles from the package (important_instructions, ignore_previous, injecagent, tool_knowledge, direct, system_message), all four suites.

Records that share a slot are not independent, so confidence intervals resample by slot for attacked records and by record for benign ones. GPU for scoring, token for Prompt-Guard.

**Outputs.** `data/agentdojo/set.json`, `reports/agentdojo_eval.csv`, `reports/agentdojo_by_attack.csv`, `reports/agentdojo_by_suite.csv`, `figures/agentdojo.png`.

In [ ]:
# === SESSION BOOTSTRAP (run first): install from the local disk, then mount ===
!pip install -q transformers datasets accelerate sentencepiece agentdojo
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
os.makedirs('reports', exist_ok=True); os.makedirs('figures', exist_ok=True); os.makedirs('data/agentdojo', exist_ok=True)
import torch, gc; print('Session ready. GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

## Stage A. Module

In [ ]:
%%writefile src/agentdojo_source.py
"""Matched guard-level evaluation set from AgentDojo: the benchmark's own tool outputs (emails, calendar entries, drive
files, Slack messages, bills, transactions, hotel and restaurant reviews) with each injection slot filled either by its
benign default or by one of AgentDojo's attack strings for one of its injection tasks. Rendering uses AgentDojo's own
environment loader and attack classes, so the injected texts are exactly what the benchmark's agents would read.
"""
from __future__ import annotations
import re, json
import numpy as np

ATTACKS = ("important_instructions", "ignore_previous", "injecagent", "tool_knowledge", "direct", "system_message")
SUITES = ("workspace", "slack", "banking", "travel")


class _DummyPipe:
    name = "gpt-4o-2024-05-13"


def _flatten(obj, path=()):
    """Yield (path, value) for every leaf of a nested dict/list structure."""
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield from _flatten(v, path + (k,))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from _flatten(v, path + (i,))
    else:
        yield path, obj


def _get(obj, path):
    for p in path: obj = obj[p]
    return obj


def _parent_record(dump, leaf_path):
    """The innermost dict containing the leaf, rendered as 'field: value' lines over its scalar fields."""
    for k in range(len(leaf_path) - 1, -1, -1):
        node = _get(dump, leaf_path[:k])
        if isinstance(node, dict):
            lines = []
            for f, v in node.items():
                if isinstance(v, (str, int, float)) and str(v).strip():
                    lines.append(f"{f}: {v}")
                elif isinstance(v, list) and v and all(isinstance(x, (str, int, float)) for x in v):
                    lines.append(f"{f}: {', '.join(str(x) for x in v)}")
            return leaf_path[:k], "\n".join(lines)
    return (), str(_get(dump, leaf_path))


def attack_texts(suite, attacks=ATTACKS):
    """{(attack, injection_task_id): text} using AgentDojo's attack classes."""
    from agentdojo.attacks.attack_registry import load_attack
    out = {}
    for a in attacks:
        try: atk = load_attack(a, suite, _DummyPipe())
        except Exception: continue
        for tid, it in suite.injection_tasks.items():
            txt = None
            for ut in suite.user_tasks.values():
                try: d = atk.attack(ut, it)
                except Exception: d = {}
                if d: txt = next(iter(d.values())); break
            if txt: out[(a, tid)] = txt
    return out


def build_agentdojo_set(suites=SUITES, attacks=ATTACKS, min_chars=40, max_per_slot=None, seed=0):
    from agentdojo.task_suite.load_suites import get_suites
    S = get_suites("v1"); rng = np.random.default_rng(seed); recs = []
    for sname in suites:
        suite = S[sname]
        defaults = suite.get_injection_vector_defaults(); slots = list(defaults)
        # sentinel environment: locate each slot's leaf and parent record
        sent = {v: f"<<SLOT:{v}>>" for v in slots}
        dump_s = suite.load_and_inject_default_environment(sent).model_dump()
        dump_b = suite.load_and_inject_default_environment({}).model_dump()
        slot_parent = {}
        for path, val in _flatten(dump_s):
            if isinstance(val, str):
                for v in slots:
                    if sent[v] in val:
                        ppath, rendered = _parent_record(dump_s, path); slot_parent[v] = (ppath, rendered)
        texts = attack_texts(suite, attacks)
        kinds = {}
        for v, (ppath, rendered_s) in slot_parent.items():
            kind = ".".join(p for p in ppath if isinstance(p, str) and not p.isdigit()) or "root"
            benign = rendered_s.replace(sent[v], str(defaults[v]))
            for w in slots:                                        # other slots in the same record get their defaults
                benign = benign.replace(sent[w], str(defaults[w]))
            if len(benign) >= min_chars:
                recs.append(dict(text=benign, label=0, suite=sname, kind=kind, slot=v, attack="", injection_task="", form="slot_benign", inj_offset=-1))
            items = list(texts.items())
            if max_per_slot and len(items) > max_per_slot:
                items = [items[i] for i in rng.choice(len(items), max_per_slot, replace=False)]
            for (a, tid), txt in items:
                att = rendered_s.replace(sent[v], txt)
                for w in slots: att = att.replace(sent[w], str(defaults[w]))
                recs.append(dict(text=att, label=1, suite=sname, kind=kind, slot=v, attack=a, injection_task=tid, form="slot_attacked", inj_offset=int(att.find(txt))))
        # extra benign records: every other record with a long scalar field, from the benign environment
        seen = {r["text"] for r in recs}
        for path, val in _flatten(dump_b):
            if isinstance(val, str) and len(val) >= min_chars:
                ppath, rendered = _parent_record(dump_b, path)
                if rendered in seen or len(rendered) < min_chars: continue
                seen.add(rendered); kind = ".".join(p for p in ppath if isinstance(p, str) and not p.isdigit()) or "root"
                recs.append(dict(text=rendered, label=0, suite=sname, kind=kind, slot="", attack="", injection_task="", form="benign_record", inj_offset=-1))
    return recs


## Stage B. Build the matched set (cached)

In [ ]:
import importlib, json, collections, numpy as np, pandas as pd
import agentdojo_source; importlib.reload(agentdojo_source)
from agentdojo_source import build_agentdojo_set, ATTACKS, SUITES
fp = 'data/agentdojo/set.json'
if os.path.exists(fp): R = json.load(open(fp)); print('set loaded from cache:', len(R))
else: R = build_agentdojo_set(); json.dump(R, open(fp, 'w')); print('set built and cached:', len(R))
WINDOW_CHARS = 1600   # conservative for a 512-token window: injections starting beyond this are invisible to every scorer
R_ALL = R; R = [r for r in R_ALL if r['label'] == 0 or r.get('inj_offset', 0) <= WINDOW_CHARS]
print(f'records with the injection beyond the window, excluded from metrics: {len(R_ALL) - len(R)} of {sum(r["label"] for r in R_ALL)} attacked')
TXT = [r['text'] for r in R]; Y = np.array([r['label'] for r in R]); SUITE = np.array([r['suite'] for r in R]); ATK = np.array([r['attack'] for r in R]); SLOT = np.array([f"{r['suite']}|{r['slot']}" for r in R])
print('by suite/label:', dict(collections.Counter((s, int(y)) for s, y in zip(SUITE, Y))))
print('by attack:', dict(collections.Counter(a for a in ATK if a)))
print('\nexample attacked record:\n', [r for r in R if r['label'] == 1][0]['text'][:500])

## Stage C. Score every scorer (cached)

In [ ]:
import structural_train; importlib.reload(structural_train)
from structural_train import score_texts, load_jsonl
from detectors import score_released
RELEASED = {'protectai_v2': 'protectai/deberta-v3-base-prompt-injection-v2', 'prompt_guard_2': 'meta-llama/Llama-Prompt-Guard-2-86M', 'prompt_guard_2_22m': 'meta-llama/Llama-Prompt-Guard-2-22M',
            'deepset_injection': 'deepset/deberta-v3-base-injection', 'protectai_v1': 'protectai/deberta-v3-base-prompt-injection', 'fmops_distilbert': 'fmops/distilbert-prompt-injection'}
LABEL = {'protectai_v2': 'ProtectAI-v2', 'prompt_guard_2': 'Prompt-Guard-2 (86M)', 'prompt_guard_2_22m': 'Prompt-Guard-2 (22M)', 'deepset_injection': 'deepset injection', 'protectai_v1': 'ProtectAI-v1', 'fmops_distilbert': 'fmops distilbert',
         'full': 'structural (full)', 'full_aug': 'structural (full_aug)', 'conventional_ft': 'conventional fine-tune', 'piguard': 'PIGuard'}
MDIR = {'full': 'data/structural/model_full/best', 'full_aug': 'data/structural/model_full_aug/best', 'conventional_ft': 'data/structural/model_conventional_ft/best'}
if any(not os.path.exists(f'data/agentdojo/score_{t}.npy') for t in RELEASED):
    from huggingface_hub import login; import getpass
    _t = getpass.getpass('HF token (gated Prompt-Guard checkpoints): ').strip()
    if _t: login(token=_t)
TXT_ALL = [r['text'] for r in R_ALL]; KEEP = np.array([i for i, r in enumerate(R_ALL) if r['label'] == 0 or r.get('inj_offset', 0) <= WINDOW_CHARS])
S = {}; failed = []
for t, mid in RELEASED.items():
    fp = f'data/agentdojo/score_{t}.npy'
    if not os.path.exists(fp):
        print('scoring', t)
        try: np.save(fp, score_released(TXT_ALL, mid))
        except Exception as e: failed.append(f'{t}: {type(e).__name__}: {str(e)[:120]}'); continue
    S[LABEL[t]] = np.load(fp)[KEEP]
for cfg, mdir in MDIR.items():
    fp = f'data/agentdojo/score_{cfg}.npy'
    if not os.path.exists(fp):
        if not os.path.exists(os.path.join(mdir, 'config.json')): print(cfg, 'model not present; skipped'); continue
        print('scoring', cfg); np.save(fp, score_texts(mdir, TXT_ALL))
    S[LABEL[cfg]] = np.load(fp)[KEEP]
pg = 'data/closers/id_piguard.json'
if os.path.exists(pg):
    fp = 'data/agentdojo/score_piguard.npy'
    if not os.path.exists(fp):
        try:
            from transformers import AutoTokenizer, AutoModelForSequenceClassification
            mid = json.load(open(pg)); tok = AutoTokenizer.from_pretrained(mid, trust_remote_code=True); mdl = AutoModelForSequenceClassification.from_pretrained(mid, trust_remote_code=True).to('cuda' if torch.cuda.is_available() else 'cpu').eval()
            idx = next((int(i) for i, l in mdl.config.id2label.items() if any(h in str(l).lower() for h in ('inject', 'malicious', 'label_1'))), mdl.config.num_labels - 1)
            out = []
            with torch.no_grad():
                for i in range(0, len(TXT_ALL), 16):
                    b = tok(TXT_ALL[i:i + 16], truncation=True, max_length=512, padding=True, return_tensors='pt').to(mdl.device); out.append(torch.softmax(mdl(**b).logits.float(), -1)[:, idx].cpu().numpy())
            np.save(fp, np.concatenate(out)); del mdl; gc.collect(); torch.cuda.empty_cache()
        except Exception as e: print('PIGuard scoring failed:', type(e).__name__, str(e)[:120])
    if os.path.exists(fp): S['PIGuard'] = np.load(fp)[KEEP]
if failed: print('FAILED:', failed)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
STR_TRAIN = load_jsonl('data/structural/train.jsonl')
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200000, sublinear_tf=True)
lr = LogisticRegression(C=1.0, max_iter=2000).fit(vec.fit_transform([r['text'] for r in STR_TRAIN]), [r['label'] for r in STR_TRAIN])
S['TF-IDF reference'] = lr.predict_proba(vec.transform(TXT))[:, 1]
for k, v in S.items(): assert len(v) == len(TXT), k
print('scorers:', list(S))

## Stage D. Metrics with slot-level bootstrap intervals

In [ ]:
from sklearn.metrics import roc_auc_score
import interval_panel; importlib.reload(interval_panel)
from interval_panel import three_thresholds, fnr, fpr, KEYS
ben = Y == 0; atk = Y == 1
def cluster_boot(fn, B=1000, seed=0):
    rng = np.random.default_rng(seed); slots = np.unique(SLOT[atk]); bidx = np.where(ben)[0]; vals = []
    for _ in range(B):
        sl = rng.choice(slots, len(slots), replace=True); ai = np.concatenate([np.where((SLOT == s) & atk)[0] for s in sl]); bi = rng.choice(bidx, len(bidx), replace=True)
        try: vals.append(fn(ai, bi))
        except Exception: pass
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))
rows, arows, srows = [], [], []
for name, p in S.items():
    yy = lambda ai, bi: np.r_[np.ones(len(ai)), np.zeros(len(bi))]
    au = roc_auc_score(Y, p); lo, hi = cluster_boot(lambda ai, bi: roc_auc_score(yy(ai, bi), np.r_[p[ai], p[bi]]))
    pa = float(roc_auc_score(Y, p, max_fpr=0.05)); T = three_thresholds(p[ben]); tg = T['guar']
    f_lo, f_hi = cluster_boot(lambda ai, bi: float((p[ai] < tg).mean()))
    rows.append(dict(model=name, n_atk=int(atk.sum()), n_ben=int(ben.sum()), AUROC=au, AUROC_lo95=lo, AUROC_hi95=hi, pAUC05=pa, **{f't_{k}': T[k] for k in KEYS},
                     FNR_lo=fnr(p[atk], T['lo']), FNR_hi=fnr(p[atk], T['hi']), FNR_guar=fnr(p[atk], tg), FNR_guar_lo95=f_lo, FNR_guar_hi95=f_hi, calFPR_guar=fpr(p[ben], tg)))
    for a in ATTACKS:
        m = atk & (ATK == a)
        if m.sum(): arows.append(dict(model=name, attack=a, n=int(m.sum()), AUROC=roc_auc_score(np.r_[np.ones(m.sum()), np.zeros(ben.sum())], np.r_[p[m], p[ben]]), FNR_guar=fnr(p[m], tg)))
    for s in SUITES:
        ma = atk & (SUITE == s); mb = ben & (SUITE == s)
        if ma.sum() and mb.sum() >= 10: srows.append(dict(model=name, suite=s, n_atk=int(ma.sum()), n_ben=int(mb.sum()), AUROC=roc_auc_score(np.r_[np.ones(ma.sum()), np.zeros(mb.sum())], np.r_[p[ma], p[mb]]), FNR_guar=fnr(p[ma], tg)))
EV = pd.DataFrame(rows); EV.to_csv('reports/agentdojo_eval.csv', index=False)
BA = pd.DataFrame(arows); BA.to_csv('reports/agentdojo_by_attack.csv', index=False)
BS = pd.DataFrame(srows); BS.to_csv('reports/agentdojo_by_suite.csv', index=False)
pd.set_option('display.width', 250)
print('=== AgentDojo tool outputs: overall (95 percent slot-level bootstrap intervals) ==='); print(EV[['model','AUROC','AUROC_lo95','AUROC_hi95','pAUC05','FNR_lo','FNR_hi','FNR_guar','FNR_guar_lo95','FNR_guar_hi95','calFPR_guar']].round(3).to_string(index=False))
print('\n=== by attack style: FNR at the guaranteed threshold ==='); print(BA.pivot_table(index='model', columns='attack', values='FNR_guar').round(3).to_string())
print('\n=== by suite: AUROC (suites with at least 10 benign records) ==='); print(BS.pivot_table(index='model', columns='suite', values='AUROC').round(3).to_string())

## Stage E. Figure

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, p in S.items():
    f, t, _ = roc_curve(Y, p); axes[0].plot(f, t, lw=2 if name == 'structural (full)' else 1.1, label=f'{name} ({roc_auc_score(Y, p):.2f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=.8); axes[0].axvline(0.01, color='grey', ls=':', lw=.8); axes[0].set_xlabel('false-positive rate on benign tool outputs'); axes[0].set_ylabel('true-positive rate on injected tool outputs'); axes[0].set_title('AgentDojo tool outputs'); axes[0].legend(fontsize=6, loc='lower right'); axes[0].grid(alpha=.3)
piv = BA.pivot_table(index='attack', columns='model', values='FNR_guar').reindex([a for a in ATTACKS if a in set(BA.attack)]); assert len(piv)
piv.plot.bar(ax=axes[1], width=.85); axes[1].set_ylim(0, 1.05); axes[1].set_ylabel('FNR at guaranteed 1 percent FPR'); axes[1].set_title('By attack style'); axes[1].legend(fontsize=6, loc='upper left', bbox_to_anchor=(1.0, 1.0)); axes[1].grid(axis='y', alpha=.3); axes[1].tick_params(axis='x', labelrotation=30, labelsize=8)
plt.tight_layout(); plt.savefig('figures/agentdojo.png', dpi=150, bbox_inches='tight'); plt.show()

## Stage F. Log and commit

In [ ]:
from reslog import log_result
summary = ('AgentDojo overall:\n' + EV[['model','AUROC','AUROC_lo95','AUROC_hi95','pAUC05','FNR_guar','FNR_guar_lo95','FNR_guar_hi95']].round(3).to_string(index=False)
           + '\n\nBy attack (FNR_guar):\n' + BA.pivot_table(index='model', columns='attack', values='FNR_guar').round(3).to_string()
           + '\n\nBy suite (AUROC):\n' + BS.pivot_table(index='model', columns='suite', values='AUROC').round(3).to_string())
log_result('nb25: AgentDojo tool-output matched set, all scorers', summary, csv_df=EV, csv_name='agentdojo_eval.csv')
print(summary)

# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py", "nb25: AgentDojo tool-output matched set (six attack styles, four suites) scored by every detector, slot-level bootstrap intervals; add src/agentdojo_source.py"], capture_output=True, text=True)
print(r.stdout); print(r.stderr)